# 02 — Speech-to-Text (ASR) and Transcription Accuracy

**Goal:** turn each call's audio into text with Whisper, then measure how accurate that text is.

**Runs on:** Google Colab with a GPU (Runtime → Change runtime type → T4 GPU). It also runs on a CPU, just much slower.

**Input:** the 25 calls in `results/subset_ids.csv` (two audio files per call: one per speaker).

**Output**
- `results/asr_segments.csv` — every Whisper segment: call, channel, speaker role, start/end time, text, confidence
- `results/asr_words.csv` — every recognized word with its own timestamps (used for speaker turns in notebook 03)
- `results/asr_model_comparison.csv` — the model-size comparison
- `results/wer_per_call.csv`, `results/wer_summary.csv` — accuracy results

**Important:** the human transcripts in the dataset are used **only as the reference** to score Whisper. They are never used as our output.

**Sections**
1. Setup
2. Load the subset and the reference text
3. Text normalization check
4. Baseline: the dataset's own (older) ASR
5. Choose the Whisper model size
6. Transcribe all calls
7. Accuracy (WER)
8. Error examples
9. Leaked or hallucinated speech check
10. Findings
11. Save and download the results

## 1. Setup

On Colab, this cell clones the repository, installs the ASR libraries and downloads the data. On a laptop it does nothing (run `scripts/download_data.py` yourself).

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ahmedalmoutiry-wq/call-center-agent-performance-analysis.git"

if IN_COLAB:
    if not os.path.exists("/content/call-center-agent-performance-analysis"):
        !git clone -q {REPO_URL} /content/call-center-agent-performance-analysis
    os.chdir("/content/call-center-agent-performance-analysis")
    # av is pinned: av 19+ removed an argument that faster-whisper 1.2.1 still uses
    !pip install -q faster-whisper==1.2.1 av==18.1.0 jiwer==4.0.0 num2words==0.5.14
    # CUDA 12 libraries for CTranslate2 (newer Colab images ship a newer CUDA version)
    !pip install -q nvidia-cublas-cu12==12.4.5.8 nvidia-cudnn-cu12==9.1.0.70
    !python scripts/download_data.py
    !python scripts/download_data.py --audio results/subset_ids.csv
    !nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(PROJECT_ROOT))

from src.asr import ASR_SETTINGS, load_whisper, preload_cuda12_libraries, transcribe_file
from src.data_loading import RESULTS_DIR, audio_path, build_utterances_table
from src.evaluation import overlap_seconds, reference_text, word_error_rate
from src.text_normalization import check_normalization, normalize_text

pd.set_option("display.max_colwidth", 120)
print("Project root:", PROJECT_ROOT)
print("ASR settings:", ASR_SETTINGS)
print("CUDA 12 libraries loaded:", preload_cuda12_libraries())

## 2. Load the subset and the reference text

For every call we have **two channels** (two audio files). The speaker role of each channel comes from the transcript, not from the folder name (see notebook 01, section 3.1).

In [ ]:
subset = pd.read_csv(RESULTS_DIR / "subset_ids.csv")
utts = build_utterances_table(subset["call_id"].tolist())
speech = utts[~utts["is_empty"] & ~utts["is_noise_only"]]   # reference segments that contain words

# One row per (call, channel): which role is on it, where its audio is, and its reference text
channels = (utts.groupby(["call_id", "channel_index"])["speaker_role"].first()
                .reset_index())
channels["audio_file"] = [audio_path(c, ch) for c, ch in zip(channels["call_id"], channels["channel_index"])]
channels["reference"] = [reference_text(speech[(speech["call_id"] == c) & (speech["channel_index"] == ch)])
                         for c, ch in zip(channels["call_id"], channels["channel_index"])]
channels["audio_exists"] = channels["audio_file"].apply(lambda p: p.exists())

print(f"Calls: {channels['call_id'].nunique()}  |  audio files: {len(channels)}  |  "
      f"missing audio: {(~channels['audio_exists']).sum()}")
channels[["call_id", "channel_index", "speaker_role", "audio_exists", "reference"]].head(4)

## 3. Text normalization check

WER compares words exactly. Without normalization, `1:30 PM` and `one thirty p m` would be counted as errors although they mean the same. Both reference and Whisper output go through **the same** rules in `src/text_normalization.py`:

- lower case, no punctuation
- numbers, times and amounts written as words (the reference style)
- remove annotation tags (`[noise]`, `<unk>`), word fragments (`wou~`) and hesitation sounds (`uh`, `um`)

The examples below are also tests: the cell fails if any rule breaks.

In [ ]:
check_normalization()
from src.text_normalization import NORMALIZATION_EXAMPLES
pd.DataFrame([(raw, normalize_text(raw)) for raw, _ in NORMALIZATION_EXAMPLES],
             columns=["raw text", "normalized"])

## 4. Baseline: the dataset's own ASR

The dataset includes an older machine transcript (`provider_asr`). Scoring it on the same calls gives a **reference point** for our Whisper results. It is not our output.

In [ ]:
provider = utts.groupby(["call_id", "channel_index"]).apply(
    lambda g: " ".join(g.sort_values("start_s")["provider_asr"]), include_groups=False
).rename("provider_asr").reset_index()
baseline_input = channels.merge(provider, on=["call_id", "channel_index"])

baseline_rows = [{"scope": "baseline (dataset ASR): overall",
                  **word_error_rate(baseline_input["reference"].tolist(), baseline_input["provider_asr"].tolist())}]
for role, group in baseline_input.groupby("speaker_role"):
    baseline_rows.append({"scope": f"baseline (dataset ASR): {role}",
                          **word_error_rate(group["reference"].tolist(), group["provider_asr"].tolist())})
baseline = pd.DataFrame(baseline_rows)
baseline

## 5. Choose the Whisper model size

Bigger Whisper models are usually more accurate but slower. Instead of guessing, we compare three sizes on 3 calls (6 audio files, one call from each of three different task types) and pick the **lowest WER**.

- `small.en`, `medium.en` — English-only models
- `large-v3` — the largest multilingual model

**Real-time factor (RTF)** = processing time ÷ audio duration. RTF 0.1 means one minute of audio takes 6 seconds.

In [ ]:
MODELS_TO_COMPARE = ["small.en", "medium.en", "large-v3"]
comparison_calls = subset.drop_duplicates("task_type")["call_id"].head(3).tolist()
comparison_files = channels[channels["call_id"].isin(comparison_calls)]
comparison_audio_s = subset.set_index("call_id").loc[comparison_calls, "duration_s"].sum() * 2

results = []
for model_size in MODELS_TO_COMPARE:
    model, device = load_whisper(model_size)
    hyps, seconds = [], 0.0
    for audio_file in comparison_files["audio_file"]:
        segs, _words, took = transcribe_file(model, audio_file)
        hyps.append(" ".join(s["asr_text"] for s in segs))
        seconds += took
    score = word_error_rate(comparison_files["reference"].tolist(), hyps)
    results.append({"model": model_size, "device": device, "wer": score["wer"],
                    "seconds": round(seconds, 1), "rtf": round(seconds / comparison_audio_s, 3)})
    del model                                   # free GPU memory before loading the next model

model_comparison = pd.DataFrame(results)
model_comparison.to_csv(RESULTS_DIR / "asr_model_comparison.csv", index=False)
print("Calls used:", comparison_calls)
model_comparison

In [ ]:
# Rule: the model with the lowest WER on the comparison calls
CHOSEN_MODEL = model_comparison.sort_values(["wer", "seconds"]).iloc[0]["model"]
print("Chosen model:", CHOSEN_MODEL)

## 6. Transcribe all calls

Each audio file is transcribed separately, so every Whisper segment already belongs to one speaker. The role comes from the channel table built in section 2.

In [ ]:
model, device = load_whisper(CHOSEN_MODEL)
print(f"Transcribing {len(channels)} files with {CHOSEN_MODEL} on {device} ...")

segment_rows, word_rows, total_seconds = [], [], 0.0
for i, ch in enumerate(channels.itertuples(), start=1):
    segs, words, took = transcribe_file(model, ch.audio_file)
    total_seconds += took
    file_info = {"call_id": ch.call_id, "channel_index": ch.channel_index, "speaker_role": ch.speaker_role}
    segment_rows += [{**file_info, **seg} for seg in segs]
    word_rows += [{**file_info, **w} for w in words]
    if i % 10 == 0 or i == len(channels):
        print(f"  {i}/{len(channels)} files  ({total_seconds:.0f} s so far)")

asr_segments = pd.DataFrame(segment_rows).sort_values(["call_id", "start_s"]).reset_index(drop=True)
asr_words = pd.DataFrame(word_rows).sort_values(["call_id", "start_s"]).reset_index(drop=True)
asr_segments.insert(0, "asr_model", CHOSEN_MODEL)
asr_words.insert(0, "asr_model", CHOSEN_MODEL)
asr_segments.to_csv(RESULTS_DIR / "asr_segments.csv", index=False)

total_audio_s = subset["duration_s"].sum() * 2
print(f"Segments: {len(asr_segments)}  |  words: {len(asr_words)}  |  RTF: {total_seconds / total_audio_s:.3f}")
asr_segments.head(8)

## 7. Accuracy (WER)

We report WER three ways:
- **Overall** — all calls and both speakers together
- **By speaker role** — the customer side matters most, because summaries and sentiment depend on it
- **Per call** — to find where Whisper struggles

In [ ]:
hyp_text = (asr_segments.groupby(["call_id", "channel_index"])["asr_text"]
            .apply(" ".join).rename("hypothesis").reset_index())
scored = channels.merge(hyp_text, on=["call_id", "channel_index"], how="left")
scored["hypothesis"] = scored["hypothesis"].fillna("")      # a channel with no Whisper output

summary_rows = [{"scope": "overall", **word_error_rate(scored["reference"].tolist(), scored["hypothesis"].tolist())}]
for role, group in scored.groupby("speaker_role"):
    summary_rows.append({"scope": f"role: {role}", **word_error_rate(group["reference"].tolist(), group["hypothesis"].tolist())})
wer_summary = pd.concat([pd.DataFrame(summary_rows), baseline], ignore_index=True)
wer_summary.to_csv(RESULTS_DIR / "wer_summary.csv", index=False)
wer_summary

In [ ]:
per_call = []
for row in scored.itertuples():
    per_call.append({"call_id": row.call_id, "speaker_role": row.speaker_role,
                     **word_error_rate([row.reference], [row.hypothesis])})
wer_per_call = pd.DataFrame(per_call).merge(subset[["call_id", "task_type", "caller_audio_quality"]], on="call_id")
wer_per_call.to_csv(RESULTS_DIR / "wer_per_call.csv", index=False)

wer_per_call.sort_values("wer", ascending=False).head(10)

## 8. Error examples

A WER number alone does not show **what** went wrong. Below: the customer side of the worst and the best call, aligned word by word.

In the alignment, `S` = substitution, `D` = deletion, `I` = insertion.

In [ ]:
import jiwer

customer = wer_per_call[wer_per_call["speaker_role"] == "caller"].dropna(subset=["wer"])
for label, call_id in [("WORST", customer.sort_values("wer").iloc[-1]["call_id"]),
                       ("BEST", customer.sort_values("wer").iloc[0]["call_id"])]:
    row = scored[(scored["call_id"] == call_id) & (scored["speaker_role"] == "caller")].iloc[0]
    out = jiwer.process_words(normalize_text(row["reference"]), normalize_text(row["hypothesis"]))
    print(f"===== {label} customer transcript: call {call_id}  (WER {out.wer:.1%}) =====")
    print(jiwer.visualize_alignment(out, show_measures=False, skip_correct=False))

## 9. Leaked or hallucinated speech check

Notebook 01 found that some audio files contain the **other speaker faintly** (echo). Whisper may also invent words in silence (hallucination). Both look the same: a word recognized at a moment when **this speaker was not talking** according to the reference timestamps.

We check every **word** (not segment: one Whisper segment can span several sentences and the pauses between them, so its time range is too coarse).

For each recognized word we ask:
1. Was **this speaker** talking at that moment? (inside one of their reference utterances, ± 0.3 s because reference times are approximate)
2. If not, was the **other speaker** talking? Then the word is probably **echo** from the other side.
3. Neither? Then it is a possible **hallucination** (or a reference utterance with missing timing).

In [ ]:
PAD_S = 0.3   # tolerance around reference utterances, in seconds

def speech_intervals(call_id, channel_index):
    s = speech[(speech["call_id"] == call_id) & (speech["channel_index"] == channel_index)]
    return list(zip(s["start_s"] - PAD_S, s["end_s"] + PAD_S))

def inside_any(t, intervals):
    return any(start <= t <= end for start, end in intervals)

own_flags, other_flags = [], []
for (call_id, channel), words in asr_words.groupby(["call_id", "channel_index"], sort=False):
    own = speech_intervals(call_id, channel)
    other = speech_intervals(call_id, 3 - channel)          # channels are 1 and 2
    mid = (words["start_s"] + words["end_s"]) / 2           # middle of each word
    own_flags += [(i, inside_any(t, own)) for i, t in zip(words.index, mid)]
    other_flags += [(i, inside_any(t, other)) for i, t in zip(words.index, mid)]

asr_words["in_own_speech"] = pd.Series(dict(own_flags))
asr_words["in_other_speech"] = pd.Series(dict(other_flags))
asr_words["has_text"] = asr_words["word"].apply(normalize_text) != ""   # ignore "Mm-hmm", punctuation ...

outside = asr_words[asr_words["has_text"] & ~asr_words["in_own_speech"]]
outside = outside.assign(likely_cause=outside["in_other_speech"].map({True: "echo of other speaker",
                                                                      False: "possible hallucination"}))
total = int(asr_words["has_text"].sum())
print(f"Recognized words: {total}  |  outside own speech: {len(outside)} ({len(outside) / total:.1%})")
outside.groupby(["speaker_role", "likely_cause"]).size().rename("words").reset_index()

In [ ]:
# Group neighbouring flagged words into short phrases, to read them easily
phrases = (outside.groupby(["call_id", "speaker_role", "segment_id", "likely_cause"])
                  .agg(start_s=("start_s", "min"), end_s=("end_s", "max"),
                       text=("word", " ".join), mean_probability=("probability", "mean"))
                  .reset_index().sort_values(["call_id", "start_s"]))
phrases.round(2).head(20)

In [ ]:
# Save the word table with the flags, so notebook 03 can use it
asr_words.to_csv(RESULTS_DIR / "asr_words.csv", index=False)

## 10. Findings

_To be written after the Colab run, from the numbers above._

## 11. Save and download the results (Colab only)

Colab files disappear when the session ends. This cell zips the result files so you can download them and copy them into your local project.

Also save the notebook with its outputs: **File → Download → Download .ipynb**.

In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import files

    export_dir = Path("/content/asr_results")
    export_dir.mkdir(exist_ok=True)
    for name in ["asr_segments.csv", "asr_words.csv", "asr_model_comparison.csv",
                 "wer_summary.csv", "wer_per_call.csv"]:
        shutil.copy(RESULTS_DIR / name, export_dir / name)
    shutil.make_archive("/content/asr_results", "zip", export_dir)
    files.download("/content/asr_results.zip")